# Spark Bronze wikpedia page reads

In [1]:

import os
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta import configure_spark_with_delta_pip

def get_config(config_name):

    config_server_url = os.environ.get("TFDS_CONFIG_URL")
    if config_server_url is None:
        config_server_url = "http://tfds-config:8005/api/configs"

    config_url = config_server_url + "/" + config_name

    print(f"retrieving {config_name} config from {config_url}")
    response = requests.get(config_url)
    response.raise_for_status()
    if response.json() is None:
        raise ValueError(f"Config '{config_name}' not found. config server response: {response.text}")
    cfg = response.json().get("config")
    if cfg is None:
        raise ValueError(f"Config '{config_name}' does not have a 'config' key. Config server response: {response.text}")

    if config_name=='s3' and "TFDS_S3_URL" in os.environ.keys():
        cfg["url"] = os.environ["TFDS_S3_URL"]
    if config_name=='spark' and "TFDS_SPARK_MASTER_URL" in os.environ.keys():
        cfg["master_url"] = os.environ["TFDS_SPARK_MASTER_URL"]
    return cfg

import pyspark
def get_spark_session():
    """Get spark client for s3."""
    s3_cfg = get_config("s3")
    spark_cfg = get_config("spark")

    print(f"using s3 endpoint: {s3_cfg['url']}")
    print(f"using spark master: {spark_cfg['master_url']}")


    # builder =  (  SparkSession
    #     .builder
    #     .master(spark_cfg['master_url'])
    #     .appName("Wikipedia page reads - Bronze")
    #     .config("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
    #     .config("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
    #     .config("spark.hadoop.fs.s3a.endpoint", s3_cfg["url"])

    #     #.config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    #     .config("spark.hadoop.fs.s3a.path.style.access", "true")
    #     #.config("spark.sql.warehouse.dir", "s3a://dwh/warehouse/")
    #     .config("spark.sql.warehouse.dir", "/opt/tfds/data/spark/warehouse/")

    #     .config("spark.executor.userClassPathFirst", "true")


    #     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    #     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    #     .config("spark.jars.packages", "io.delta:delta-spark_2.12:2.4.0,io.delta:delta-storage:2.4.0,org.apache.hadoop:hadoop-aws:3.3.4, org.apache.hadoop:aws-java-sdk-bundle:1.12.262, org.apache.hadoop:hadoop-common:3.3.4")

    #     # .config("spark.jars",  "../custom_jars/hadoop-aws-3.4.1.jar,../custom_jars/bundle-2.31.25.jar")
    #     .config("spark.executor.memory", "2g")
    # )
    # builder = configure_spark_with_delta_pip(builder)
    # spark_session = builder.getOrCreate()


    conf = (
        pyspark.conf.SparkConf()
        .setAppName("WhenDoIGetToSpark")
        .set(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .set("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
        .set("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
        .set("spark.hadoop.fs.s3a.endpoint", s3_cfg["url"])
        .set("spark.hadoop.fs.s3a.path.style.access", "true")
        .set("spark.sql.warehouse.dir", "s3a://dwh/warehouse/")
        .set("spark.sql.shuffle.partitions", "12")
        .set("spark.executor.memory", "2g")
        .setMaster(spark_cfg['master_url'])
        #.setMaster("local[*]")
    )

    extra_packages = [
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "org.apache.hadoop:hadoop-common:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    ]

    builder = pyspark.sql.SparkSession.builder.appName("MyApp").config(conf=conf)
    spark_session = configure_spark_with_delta_pip(
        builder, extra_packages=extra_packages
    ).getOrCreate()


    return spark_session

def show_cfg(spark_session):
    cfg = spark_session.sparkContext.getConf().getAll()
    for key, value in cfg:
        if key in (
            'spark.submit.pyFiles',
            'spark.driver.extraJavaOptions',
            'park.app.initial.jar.urls',
            'spark.files',
            'spark.repl.local.jars',
            'spark.app.initial.file.urls'
            'spark.executor.extraJavaOption',
            'spark.app.initial.jar.urls'
            'spark.app.initial.file.urls'
            ):
            print(key)
            for l in value.split(','):
                print('    ' + str(l))
        else:
            print(f'{key} = {value}')


In [2]:
from pyspark.sql.functions import input_file_name, col, sum as _sum, substring, to_date

def load_page_reads(s3_path):
    schema = StructType([
        StructField(name="domain_code", dataType=StringType(), nullable = True),
        StructField("page_title", StringType(), True),
        StructField("count_views", StringType(), True)
    ])
    spark.sparkContext.setLogLevel("WARN")


    print(f'Loading data:{s3_path}')
    data = (
        spark.read.format("csv")
        .option("delimiter", " ")
        .option("header", "false")
        .option("inferSchema", "false")
        .schema(schema)
        .load(s3_path)
    )
    print('Files loaded:')
    for f in data.inputFiles():
        print(f)
    return data

def enrich_page_reads(data):
    data_enriched = (
        data
        .na.drop(subset=["domain_code"])
        .filter(~col("page_title").contains(":"))
        .filter(~col("page_title").isin("-", 'Main_Page', 'Forside', 'Hauptseite', 'wiki.phtml'))
        .withColumn("country_code", substring(col("domain_code"), 1, 2))
        .filter(col("domain_code").isin("sv", 'dk', 'no', 'de', 'en'))
        .withColumn("count_views", col("count_views").cast(IntegerType()))
        .withColumn("file_name", input_file_name())
        .withColumn("date", to_date(substring(col("file_name"), -18, 8), 'yyyyMMdd'))
    )
    return data_enriched

def write_page_reads(data):
    table = "bronze.wikipedia_page_reads"
    print(f'Writing data to {table}')
    (   data
        .write
        .mode("overwrite")   # Options: 'overwrite', 'append', 'ignore', 'error' (default)
        .option("mergeSchema", "true")
        .format("delta")    # Options: 'parquet', 'csv', 'json', 'orc', etc.
        .partitionBy("date")
        .saveAsTable(table)
    )


spark = get_spark_session()
spark.sparkContext.setLogLevel("WARN")

# spark.catalog.clearCache()
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/11/pageviews-20250311-220000.gz"
s3_path = "s3a://data/test.csv"
s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/*/*.gz"

data_raw = load_page_reads(s3_path=s3_path)
data_enriched = enrich_page_reads(data_raw)
write_page_reads(data=data_enriched)
spark.stop()
print('all done')


retrieving s3 config from http://tfds-config:8005/api/configs/s3
retrieving spark config from http://tfds-config:8005/api/configs/spark
using s3 endpoint: http://s3-minio:9000
using spark master: spark://spark-master:7077


25/04/21 17:09:24 WARN Utils: Your hostname, McJens.local resolves to a loopback address: 127.0.0.1; using 192.168.0.152 instead (on interface en0)
25/04/21 17:09:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/jens/.ivy2/cache
The jars for the packages stored in: /Users/jens/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.hadoop#hadoop-common added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8a950f52-4e06-4b3f-bed0-abcd0219e1b9;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/jens/src/the-free-data-stack/.venv/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.hadoop#hadoop-common;3.3.4 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-protobuf_3_7;1.1.1 in central
	found org.apache.hadoop#hadoop-annotations;3.3.4 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.1.1 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sn

Loading data:s3a://data/wikipedia_pageviews/2025/2025-03/*/*.gz
Files loaded:
s3a://data/wikipedia_pageviews/2025/2025-03/11/pageviews-20250311-220000.gz
s3a://data/wikipedia_pageviews/2025/2025-03/11/pageviews-20250311-230000.gz
s3a://data/wikipedia_pageviews/2025/2025-03/12/pageviews-20250312-000000.gz
Writing data to bronze.wikipedia_page_reads


25/04/21 17:09:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


all done


## Bronze
Performs: 
* ingestion
* column naming
* column casting
* get the date from the filename
* partitioning

In [ ]:
from pyspark.sql.functions import input_file_name, col, sum as _sum, substring, to_date

spark = get_spark_session()

s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/**/*.gz"
s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/11/pageviews-20250311-220000.gz"

if not s3_path.startswith("s3a://"):
    s3_path =  "s3a://" + s3_path

schema = StructType([
    StructField(name="domain_code", dataType=StringType(), nullable = True),
    StructField("page_title", StringType(), True),
    StructField("count_views", StringType(), True),
    # StructField("total_response_size", StringType(), True), this eems not to be populated, discard it
])
spark.sparkContext.setLogLevel("DEBUG")
print(f"Reading data from {s3_path}")

# df_base = (
#     spark.read.format("csv")
#     .option("delimiter", " ")
#     .option("header", "false")
#     .option("inferSchema", "false")
#     .schema(schema)
#     .load(s3_path)
# )

# df_base = (
#     df_base
#     .na.drop(subset=["domain_code"])
#     .filter(~col("page_title").contains(":"))
#     .filter(~col("page_title").isin("-", 'Main_Page', 'Forside', 'Hauptseite', 'wiki.phtml'))
#     .withColumn("country_code", substring(col("domain_code"), 1, 2))
#     .filter(col("domain_code").isin("sv", 'dk', 'no', 'de', 'en'))
#     .withColumn("count_views", col("count_views").cast(IntegerType()))
#     .withColumn("file_name", input_file_name())
#     .withColumn("date", to_date(substring(col("file_name"), -18, 8), 'yyyyMMdd'))
# )
# # .repartition("date").cache()

df_base = spark.sparkContext.range(2000)
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

(   df_base
    .write
    .mode("overwrite")   # Options: 'overwrite', 'append', 'ignore', 'error' (default)
    .format("parquet")    # Options: 'parquet', 'csv', 'json', 'orc', etc.
    # .partitionBy("date")
    .saveAsTable("bronze.wikipedia_page_reads2")
)

In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

aggregated_df = (
    spark.table("bronze.wikipedia_page_reads")
    .groupBy('date', "page_title", "country_code")
    .agg(
        _sum(col("count_views")).alias("total_count_views"),
    )
)

national_win = (
    Window
    .partitionBy('date', "country_code")
    .orderBy(col("total_count_views").desc())
)

ranked_df = (
    aggregated_df
    .withColumn("national_rank", row_number().over(national_win))
    )

final_df = (
    ranked_df
    .filter(col('national_rank') <= 3)
    .orderBy('date', "national_rank", "country_code")
)

# final_df.explain(mode="extended")

In [ ]:
final_df.show(50, truncate=False)

In [ ]:
# ranked_df.explain(mode="extended")
# spark.stop()